# Requirement 7 · The classroom app: detection, counting, per-student states, trends, convolution

Everything is measured on the Kaggle **test classroom** (taimoor-zaeem), which no model saw during training:
YOLOv8 / Faster R-CNN counting, YOLOv8n fine-tuned on the training classrooms, ResNet18 and GoogLeNet fine-tuned for
attentive / distracted / inactive, the trend over the video, and 2D/3D convolution figures.
See `docs/guide/07_classroom_app.md`.

**Use a GPU runtime** (Runtime → Change runtime type → T4 GPU): fine-tuning takes minutes instead of an hour.

## 0 · Put the project on Google Drive (once)

1. Zip the `smart Attendance` folder on your laptop **without** `data/`, `models/` and `reports/`
   (they are rebuilt here). Rename the zip to `smart_attendance.zip`.
2. Upload it to `MyDrive/` in Google Drive.

Keeping data and results on Drive means they survive when Colab disconnects (Colab's own disk is wiped).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
PROJECT = '/content/drive/MyDrive/smart_attendance'
if not os.path.exists(PROJECT):
    zipfile.ZipFile('/content/drive/MyDrive/smart_attendance.zip').extractall(PROJECT)
# if the zip contained a top-level folder, step into it
inner = [d for d in os.listdir(PROJECT) if os.path.isfile(os.path.join(PROJECT, d, 'pyproject.toml'))]
if inner:
    PROJECT = os.path.join(PROJECT, inner[0])
%cd "$PROJECT"
!ls

In [ ]:
# Colab already has torch, torchvision and OpenCV; this adds the rest and installs our package
!pip install -q -r requirements.txt ultralytics kagglehub
!pip install -q -e .

In [ ]:
import os, kagglehub
if not os.path.exists('data/processed/kaggle_classroom/crops'):
    from smartattend.data.kaggle_classroom import prepare
    prepare('data/raw', 'data/processed/kaggle_classroom',
            source=kagglehub.dataset_download('mustafaxgm/classroom-student-engagement-dataset'))

## 1 · Fine-tune YOLOv8n on the training classrooms (people only)

In [ ]:
import torch
from smartattend.vision.finetune_yolo import finetune
device_note = 'GPU' if torch.cuda.is_available() else 'CPU (slow)'
print('training on', device_note)
s = finetune('data/processed/kaggle_classroom/yolo_person', 'models', 'reports/experiments/detection',
             epochs=15, run_name='yolov8n_kaggle', target_name='yolov8n-kaggle.pt')
print(s['metrics_val'])

## 2 · Evaluate everything on the test classroom

In [ ]:
!python -m smartattend classroom

In [ ]:
from IPython.display import Image, display
for name in ['classroom_test_eval.png', 'classroom_test_annotated.jpg', 'conv2d_explained.png', 'conv3d_motion.png',
             'resnet_filters_featuremaps.png']:
    display(Image(f'reports/figures/vision/{name}'))

## 3 · The live app

In [ ]:
# Streamlit needs a public tunnel on Colab; on your laptop just run:
#   python -m streamlit run app/streamlit_app.py
!python -m smartattend analyze --input data/processed/kaggle_classroom/yolo_person/images/test --out reports/session_test --no-video
!head -5 reports/session_test/engagement_timeline.csv

## 4 · Tests

In [ ]:
!python -m pytest -q tests/test_classroom_eval.py tests/test_vision_sequence.py